# Government Expenditure Fraud Detection System (Data Generation)
This notebook generates a synthetic dataset of 10,000+ transactions to test a multi-layered audit system.
- **Goal**: Infill specific anomalies to validate SQL and Machine Learning detection models.
- **Scenarios**: Short-term stringing, Long-term monopoly, and Q4 budget dumping.

In [1]:
from datetime import datetime, timedelta
import os
import numpy as np
import pandas as pd

# Setting seed for reproducibility
np.random.seed(42)
n_rows = 10000
start_date = datetime(2018, 1, 1)
end_date = datetime(2024, 12, 31)

# WA State BARS Category IDs (1:General ~ 8:Misc)
dept_ids = list(range(1, 9))

In [2]:
# Define paths
from pathlib import Path
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "02_data"
OUTPUT_PATH = BASE_DIR / "03_outputs"
MODEL_PATH = BASE_DIR / "06_models"

MODEL_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

### 1. Baseline Data (Normal Operations)
Generating standard operational expenses that comply with the **$40,000 Direct Buy Limit**. 
These transactions serve as the "control group" for our audit queries.

In [3]:
data = []
for i in range(n_rows):
    dt = start_date + timedelta(days=np.random.randint(0, (end_date - start_date).days))
    dept_id = np.random.choice(dept_ids)
    amount = round(np.random.uniform(100, 15000), 2) # Normal range
    vendor = f"Vendor_{np.random.randint(1, 100)}"
    data.append([i+1, dt.strftime('%Y-%m-%d'), dept_id, vendor, amount, "Standard operational expense"])

df = pd.DataFrame(data, columns=['tx_id', 'exp_date', 'dept_id', 'vendor_name', 'amount', 'description'])

### 2. Scenario A: Tactical Anomaly
**Target**: SQL Step 1 (Window Function)
**Logic**: Split a single project into two payments within a 7-day period to bypass the $40k bidding threshold.

In [4]:
for i in range(5):
    base_idx = np.random.randint(0, n_rows)
    target_date = datetime.strptime(df.loc[base_idx, 'exp_date'], '%Y-%m-%d')
    df.loc[base_idx, ['vendor_name', 'amount', 'description']] = ["Target_Vendor_A", 21000.00, "Project Alpha Phase 1"]
    
    new_row = {
        'tx_id': df['tx_id'].max() + 1,
        'exp_date': (target_date + timedelta(days=3)).strftime('%Y-%m-%d'),
        'dept_id': df.loc[base_idx, 'dept_id'],
        'vendor_name': "Target_Vendor_A",
        'amount': 22000.00,
        'description': "Project Alpha Phase 2"
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

### 3. Scenario B: Strategic Anomaly
**Target**: SQL Step 2 (Annual Cumulative)
**Logic**: Payments are spaced 4 months apart to evade rolling window detection but exceed $40k annually.

In [5]:
for i in range(3):
    dept_id = 4 # Transportation
    year = 2023
    for month_offset in [0, 4, 8]:
        new_row = {
            'tx_id': df['tx_id'].max() + 1,
            'exp_date': datetime(year, 1 + month_offset, 15).strftime('%Y-%m-%d'),
            'dept_id': dept_id,
            'vendor_name': "Monopoly_Corp_B",
            'amount': 39000.00,
            'description': "Strategic Maintenance Contract"
        }
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

### 4. Scenario C: Administrative Anomaly
**Target**: SQL Step 3 (Q4 Ratio)
**Logic**: Department 6 (Social Services) shows a massive spending spike in Q4 compared to the first 9 months.

In [6]:
dept6_indices = df[df['dept_id'] == 6].index
for idx in dept6_indices:
    orig_date = datetime.strptime(df.loc[idx, 'exp_date'], '%Y-%m-%d')
    if orig_date.month <= 9:
        new_q4_date = datetime(orig_date.year, np.random.randint(10, 13), np.random.randint(1, 28))
        df.loc[idx, 'exp_date'] = new_q4_date.strftime('%Y-%m-%d')
        df.loc[idx, 'amount'] = df.loc[idx, 'amount'] * 6
        df.loc[idx, 'description'] = "Emergency Year-end Fund Utilization"

### 5. Result Export
Saving the dataset to CSV for **SQL database** import (DBeaver).

In [7]:
FILE_PATH = os.path.join(DATA_PATH, 'expenditure_data.csv')
df.to_csv(FILE_PATH, index=False)

print(f"✅ Dataset generated: {FILE_PATH}")
print(f"📊 Final Shape: {df.shape}")

✅ Dataset generated: C:\Users\tjrdu\OneDrive\Desktop\GitHub\Financial-Risk-Decision-Intelligence-System\02_data\expenditure_data.csv
📊 Final Shape: (10014, 6)
